## Recommender System - Product Gap Highlighter (Same Month Last Year vs Last Month)

In [1]:
import pandas as pd
from datetime import datetime

### Load Data

In [2]:
prev_df = pd.read_excel(r"D:\OneDrive - Nilons Enterprises Pvt Ltd\Desktop\Anadi\Data\YTD 2024-2025 NC_E.xlsx")
curr_df = pd.read_excel(r"D:\OneDrive - Nilons Enterprises Pvt Ltd\Desktop\Anadi\Data\SAP25_apr1st_may31st_E.xlsx")

In [3]:
prev_df.columns = prev_df.columns.str.strip()
curr_df.columns = curr_df.columns.str.strip()

### Data Preprocessing

In [4]:
prev_df['Invoice Date'] = pd.to_datetime(prev_df['Invoice Date'])
curr_df['Invoice Date'] = pd.to_datetime(curr_df['Invoice Date'])

In [5]:
prev_df.rename(columns={'Billing Amount': 'Bill Amount'}, inplace=True)
curr_df.rename(columns={'C. No': 'Distributor Code','C. Name': 'Distributor Name','C. Area': 'Area'}, inplace=True)

In [6]:
channel_map = {'EXP': 'Export','RL': 'Institutional','INST': 'Institutional','GT': 'GT','MT': 'MT','PL': 'Private Label','SMT': 'SMT','GOVT': 'Command','E-COM': 'E-Commerce','GT HO': 'Horeca'}
category_map = {'VERMICELLI-ROASTED': 'ROAST VERMICELLI','VERMICELLI-CUT': 'CUT VERMICELLI','TOOTY FRUTI': 'TOOTY FRUITY','RE 1 & 2': 'PICKLE-RE 1&2','BLENDED - WESTERN': 'SPICE-WESTERN BLEND','BLENDED - INDIAN': 'SPICE-INDIAN BLEND','SPICES-BLENDED': 'SPICE-BASIC','SPICES-CTC': 'SPICE-CTC','SPICES-RTC': 'SPICE-RTC'}
area_map = {'ORISSA': 'ODISHA','UTTRAKHAND': 'UTTARAKHAND','BAREILY': 'BAREILLY','PUNE & GOA': 'PUNE','GUJARAT-RAJKOT': 'RAJKOT','GUJARAT-AHMEDABAD': 'AHMEDABAD','CHHATISGARH': 'CHHATTISGARH','BIHAR-MUZAFFARPUR (N)': 'NORTH BIHAR','BIHAR-MUZAFFARPUR (J)': 'SOUTH BIHAR','BIHAR-PATNA': 'NORTH BIHAR','ROM 1': 'ROM','MODERN TRADE': 'MT','PRIVATE LABLE': 'Private Label'}

In [7]:
prev_df['Distribution Channel'] = prev_df['Distribution Channel'].replace(channel_map)
prev_df['Category'] = prev_df['Category'].replace(category_map)
prev_df['Area'] = prev_df['Area'].replace(area_map)

In [8]:
prev_df['Distributor Code'] = prev_df['Distributor Code'].astype(str)
curr_df['Distributor Code'] = curr_df['Distributor Code'].astype(str)

In [9]:
today = pd.to_datetime("2025-05-01")  # Assume current date is May 2025
last_month = (today.replace(day=1) - pd.DateOffset(days=1)).strftime('%Y-%m')  # Apr 2025
last_year_same_month = (today.replace(year=today.year - 1)).strftime('%Y-%m')  # May 2024

In [10]:
prev_df['period'] = prev_df['Invoice Date'].dt.to_period('M').astype(str)
curr_df['period'] = curr_df['Invoice Date'].dt.to_period('M').astype(str)

### Analysis: Identify Product Gaps

In [11]:
# Products sold last year same month
last_year_sales = prev_df[prev_df['period'] == last_year_same_month][['Distributor Code', 'Item Code']].drop_duplicates()
last_year_sales['sold_last_year_same_month'] = 1

In [12]:
# Products sold last month
last_month_sales = curr_df[curr_df['period'] == last_month][['Distributor Code', 'Item Code']].drop_duplicates()
last_month_sales['sold_last_month'] = 1

In [13]:
# Merge to detect gaps
gap_check = last_year_sales.merge(last_month_sales, on=['Distributor Code', 'Item Code'], how='left')
gap_check['sold_last_month'] = gap_check['sold_last_month'].fillna(0)

In [14]:
# Highlight products sold last year same month but NOT last month
gaps = gap_check[gap_check['sold_last_month'] == 0].copy()

In [15]:
# Add Item Name for reference
item_lookup = pd.concat([prev_df[['Item Code', 'Item Name']], curr_df[['Item Code', 'Item Name']]]).drop_duplicates()
gaps = gaps.merge(item_lookup, on='Item Code', how='left')

In [16]:
gaps = gaps[['Distributor Code', 'Item Code', 'Item Name']].sort_values(by=['Distributor Code', 'Item Code'])
display(gaps.head(20))

,Distributor Code,Item Code,Item Name
11607,100006,1400919.0,15 KG 5-6 MM PAPAYA FRUIT PRESERVED RED POU (K...
11608,100006,1400919.0,15Kg*1 5-6MM CANDIED FRUIT RED POU KRCHI
4602,100007,1400845.0,15 KG MODEL PAPAYA FRUIT PRESERVE MIX POU
4603,100007,1400845.0,15Kg*1 MODEL CANDIED FRUIT MIX POU
11612,100013,1400919.0,15 KG 5-6 MM PAPAYA FRUIT PRESERVED RED POU (K...
11613,100013,1400919.0,15Kg*1 5-6MM CANDIED FRUIT RED POU KRCHI
11915,100015,1400018.0,200 GM STD CHILLI PICKLE POU M/O
11916,100015,1400018.0,200g*40 STD CHILLI PICKLE POU MO
11919,100015,1400022.0,400 GM STD MIX PICKLE BTL M/O
11920,100015,1400022.0,400g*20 STD MIX PICKLE BTL MO
